In [ ]:
!pip install google-adk requests
!pip install googlemaps
!pip install google-cloud-aiplatform[adk,agent_engines]


In [ ]:
import getpass
import os
from google.colab import userdata
import requests
import googlemaps
from typing import Dict
from google.adk.agents import Agent


# Securely prompt for your Google Maps API key
maps_key = getpass.getpass("Enter your Google Maps API Key (starts with AIzaSy...): ")
os.environ["GOOGLE_MAPS_API_KEY"] = maps_key



/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


Enter your Google Maps API Key (starts with AIzaSy...): ··········


In [ ]:
import requests
from typing import Dict, Any

import googlemaps
from typing import Dict

def get_coordinates(location_name: str) -> Dict[str, float]:
    """
    Converts a city/location name into latitude and longitude coordinates.
    """
    import os
    from google.colab import userdata

    # Securely retrieve your Google Cloud Maps API key from Colab Secrets
    # (Make sure to save your key under the name 'GOOGLE_MAPS_API_KEY' in Colab)
    gmaps_key = os.environ.get("GOOGLE_MAPS_API_KEY")
    gmaps = googlemaps.Client(key=gmaps_key)

    # Geocode the location name
    geocode_result = gmaps.geocode(location_name)

    if not geocode_result:
        raise ValueError(f"Google Maps could not find coordinates for: {location_name}")

    location = geocode_result[0]['geometry']['location']

    return {
        "lat": float(location["lat"]),
        "lon": float(location["lng"])
    }

def get_nws_weather(lat: float, lon: float) -> str:
    """
    Fetches the real-time weather forecast from the National Weather Service API
    using latitude and longitude.
    """
    headers = {'User-Agent': 'ADK-Weather-Agent-Tutorial'}

    # Step 1: Get the NWS grid point URL for these coordinates
    points_url = f"https://api.weather.gov/points/{lat},{lon}"
    points_res = requests.get(points_url, headers=headers).json()

    if "properties" not in points_res:
        return "Error: Location might be outside NWS coverage jurisdiction (US only)."

    # Step 2: Extract the specific forecast URL
    forecast_url = points_res["properties"]["forecast"]

    # Step 3: Fetch the actual weather periods
    forecast_res = requests.get(forecast_url, headers=headers).json()
    periods = forecast_res["properties"]["periods"]

    # Format the first couple of periods (e.g., Today and Tonight) into text
    forecast_summary = ""
    for period in periods[:2]:
        forecast_summary += f"**{period['name']}**: {period['detailedForecast']}\n"

    return forecast_summary

In [ ]:
weather_agent = Agent(
    name="Weather_Bot",
    #model="gemini-2.0-flash", # pick one yourself
    description="An agent that asks for a location and gives real-time weather info.",
    instruction=(
        "You are a friendly weather assistant. If the user does not provide a location, "
        "always politely ask them for it first. Once you have a location, use the "
        "`get_coordinates` tool to get lat/lon, then feed those into the `get_nws_weather` "
        "tool to fetch the forecast. Present the results clearly to the user. Note that "
        "the weather tool only supports US locations."
    ),
    tools=[get_coordinates, get_nws_weather]
)


In [ ]:
import vertexai
from vertexai.preview import reasoning_engines

# 1. Initialize global cloud properties (Required for AdkApp backend)
# Replace with your actual GCP details
PROJECT_ID = "qwiklabs-gcp-01-d59f380c208c"
LOCATION = "us-central1"

vertexai.init(project=PROJECT_ID, location=LOCATION)

# 2. Host the weather agent inside the Reasoning Engine container
app = reasoning_engines.AdkApp(
    agent=weather_agent,
    enable_tracing=False,
)
print("Reasoning engine application successfully initialized.")


Reasoning engine application successfully initialized.


In [ ]:
import time
from IPython.display import Markdown, display

# 1. Setup universal user metadata identifier
user_id = "bootcamp-test-user"

# 2. Define the set of target US cities
test_cities = [
    "Dunedin, Florida",
    "Seattle, Washington",
    "Austin, Texas",
    "Anchorage, Alaska",
    "Chicago, Illinois"
]

print(f"🚀 Starting automated weather agent verification batch...\n")

for city in test_cities:
    print(f"--------------------------------------------------")
    print(f"🔍 Testing Agent Query for: {city}")
    print(f"--------------------------------------------------")

    try:
        # Step 3: Register a clean session context inside the local cache
        new_session = app.create_session(user_id=user_id)

        # FIXED: Extract session ID using dictionary syntax instead of object attribute syntax
        session_id = new_session["id"] if isinstance(new_session, dict) else new_session.id

        print(f"🤖 Registered Session Reference: {session_id}")
        print("🤖 Agent Response:\n")

        # Step 4: Stream the response utilizing the extracted session ID string
        response_stream = app.stream_query(
            user_id=user_id,
            session_id=session_id,
            message=f"What is the current weather forecast for {city}? Please mention the city name in your answer."
        )

        # Step 5: Process streaming tokens via your custom dictionary parsing strategy
        for event in response_stream:
            try:
                # Target the exact dictionary structure you provided
                if isinstance(event, dict) and "content" in event:
                    # In some ADK payloads, parts is a list. Let's make sure it handles lists or nested dicts safely
                    parts = event["content"]["parts"]
                    text_chunk = parts[0]["text"] if isinstance(parts, list) else parts["text"]
                    if text_chunk:
                        display(Markdown(text_chunk))

                # Handle structural fallback instances if the SDK modifies object attributes
                elif hasattr(event, "content") and event.content:
                    if hasattr(event.content, "parts") and event.content.parts:
                        text_chunk = event.content.parts[0].text if isinstance(event.content.parts, list) else event.content.parts.text
                        if text_chunk:
                            display(Markdown(text_chunk))
            except (KeyError, IndexError, AttributeError):
                # Gracefully catch and ignore internal JSON tool triggers or warnings
                pass

        print("\n")  # Spacing between separate lookups

    except Exception as e:
        print(f"❌ Error testing {city}: {e}\n")

    # Prevent hitting strict rate limits on the National Weather Service API
    time.sleep(2)

print("✅ All test queries have been executed successfully.")


🚀 Starting automated weather agent verification batch...

--------------------------------------------------
🔍 Testing Agent Query for: Dunedin, Florida
--------------------------------------------------
🤖 Registered Session Reference: 4b5a4d89-99b2-4d1e-9a39-de68c340d6a7
🤖 Agent Response:



The current weather forecast for Dunedin, Florida is:

**This Afternoon**: Isolated showers and thunderstorms before 2pm. Mostly sunny, with a high near 93. Heat index values as high as 110. West wind 5 to 8 mph. Chance of precipitation is 20%. New rainfall amounts less than a tenth of an inch possible.
**Tonight**: Mostly clear, with a low around 81. Heat index values as high as 107. Southwest wind 2 to 7 mph.



--------------------------------------------------
🔍 Testing Agent Query for: Seattle, Washington
--------------------------------------------------
🤖 Registered Session Reference: ce75befe-e5ad-48b7-8f16-c73c8b9dfc1c
🤖 Agent Response:



Here is the current weather forecast for Seattle, Washington:

**Today**: Mostly cloudy, with a high near 73. Northeast wind around 5 mph.
**Tonight**: Mostly cloudy, with a low around 56. North northeast wind 2 to 8 mph.



--------------------------------------------------
🔍 Testing Agent Query for: Austin, Texas
--------------------------------------------------
🤖 Registered Session Reference: 15bbd04f-3159-45c6-9625-ca2eb666cb73
🤖 Agent Response:



...



--------------------------------------------------
🔍 Testing Agent Query for: Anchorage, Alaska
--------------------------------------------------
🤖 Registered Session Reference: fcf15f8c-a1a7-4090-af36-f9869fc5fded
🤖 Agent Response:



The current weather forecast for Anchorage, Alaska is:

**Today**: A chance of rain showers. Mostly cloudy, with a high near 65. North wind around 5 mph. Chance of precipitation is 50%.
**Tonight**: A chance of rain showers before 1am. Mostly cloudy, with a low around 50. South wind around 5 mph. Chance of precipitation is 30%.



--------------------------------------------------
🔍 Testing Agent Query for: Chicago, Illinois
--------------------------------------------------
🤖 Registered Session Reference: e294d8dd-eb1c-4002-b8c9-b6bc731fd4c1
🤖 Agent Response:



The current weather forecast for Chicago, Illinois is:

**Today**: Sunny, with a high near 89. South southwest wind 5 to 10 mph.
**Tonight**: Partly cloudy, with a low around 74. South southwest wind 5 to 10 mph.



✅ All test queries have been executed successfully.
